# SGLang 与 vLLM 核心差异 — 通俗讲解

**定位：以通俗易懂的方式对比 SGLang 与 vLLM 两大推理框架的核心差异，帮助读者根据实际场景选择合适的工具。**

> **SGLang 默认启动命令：**
> ```bash
> python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
> ```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台。*

---

## 硬件与环境要求

| 项目 | 最低要求 |
|------|----------|
| **GPU** | NVIDIA GPU（算力 ≥ 7.0） |
| **显存（VRAM）** | ≥ 6 GB（Qwen2.5-0.5B） |
| **驱动** | NVIDIA Driver ≥ 525 |
| **CUDA** | ≥ 12.1 |
| **Python** | 3.9 – 3.12 |
| **操作系统** | Linux x86_64 最佳；Windows 建议使用 WSL2 |

---

## 依赖安装

```bash
# 安装 SGLang 全部组件及 OpenAI SDK
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

> **注意：** 如需同时体验 vLLM，建议在 **独立的虚拟环境** 中安装，避免依赖冲突。
> ```bash
> # 在另一个虚拟环境中安装 vLLM
> pip install vllm "openai>=1.0.0"
> ```

---

## 使用指南

1. 本 Notebook 以 **对比讲解** 为主，代码演示部分仅启动 SGLang
2. 按顺序阅读每个单元格，理解两个框架的异同
3. 代码单元格可直接运行体验 SGLang 的功能
4. 执行完毕后务必运行「清理资源」单元格释放 GPU 显存

---

In [ ]:
# === 环境检查 ===
import subprocess  # 导入子进程模块，用于执行系统命令
import sys  # 导入系统模块，获取 Python 版本信息

print("=" * 50)  # 打印分隔线
print("环境检查")  # 打印标题
print("=" * 50)  # 打印分隔线

# 检查 Python 版本
print(f"Python 版本: {sys.version}")  # 输出当前 Python 版本

# 检查 GPU 是否可用
try:
    result = subprocess.run(  # 执行 nvidia-smi 命令查询 GPU 信息
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],  # 查询 GPU 名称和显存
        capture_output=True, text=True, timeout=10  # 捕获输出，设置超时
    )
    if result.returncode == 0:  # 如果命令执行成功
        print(f"GPU 信息: {result.stdout.strip()}")  # 输出 GPU 信息
    else:
        print("警告: nvidia-smi 执行失败")  # 输出警告
except FileNotFoundError:  # 如果找不到 nvidia-smi
    print("警告: 未找到 nvidia-smi，请确认已安装 NVIDIA 驱动")  # 输出警告

# 检查 sglang 是否已安装
try:
    import sglang  # 尝试导入 sglang
    print(f"SGLang 版本: {sglang.__version__}")  # 输出 sglang 版本
except ImportError:  # 如果未安装
    print("SGLang 未安装，请运行下方安装单元格")  # 提示安装

# 检查 openai 包
try:
    import openai  # 尝试导入 openai
    print(f"OpenAI SDK 版本: {openai.__version__}")  # 输出 openai 版本
except ImportError:  # 如果未安装
    print("OpenAI SDK 未安装")  # 提示安装

# 检查 vllm 是否已安装（可选）
try:
    import vllm  # 尝试导入 vllm
    print(f"vLLM 版本: {vllm.__version__}")  # 输出 vllm 版本
except ImportError:  # 如果未安装
    print("vLLM 未安装（本 Notebook 主要演示 SGLang，vLLM 安装为可选）")  # 提示可选

print("=" * 50)  # 打印分隔线

In [ ]:
# === 可选：安装依赖（已安装可跳过） ===
# !pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"  # 取消注释以安装 SGLang 依赖
# !pip install vllm  # 取消注释以安装 vLLM（建议在独立虚拟环境中安装）

## SGLang 与 vLLM：两大推理框架对比

**SGLang** 和 **vLLM** 是当前最热门的两个开源大模型推理框架，各有特色：

- **vLLM**（2023 年发布）：由 UC Berkeley 团队开发，凭借 **PagedAttention** 技术一举成名，是最早实现高效 KV Cache 管理的推理引擎之一，生态成熟，社区庞大。

- **SGLang**（2024 年发布）：同样由 UC Berkeley 团队成员参与，定位为「高性能推理引擎 + 前端编程语言」，核心创新是 **RadixAttention**（基数树前缀缓存）和结构化生成（JSON/正则约束），在多轮对话和共享前缀场景下表现突出。

两者都支持 **OpenAI 兼容 API**，迁移成本很低。

## 核心架构差异

| 维度 | SGLang | vLLM |
|------|--------|------|
| **定位** | 高性能推理引擎 + 前端语言 | 高性能推理引擎 |
| **KV Cache 管理** | RadixAttention（基数树自动前缀缓存） | PagedAttention（分页显存管理） |
| **调度策略** | 连续批处理 + RadixAttention 调度 | 连续批处理 |
| **API 兼容** | OpenAI 兼容 | OpenAI 兼容 |
| **前端语言** | 有（SGLang DSL） | 无 |
| **结构化输出** | 原生支持 JSON/Regex 约束 | 通过 guided decoding 支持 |
| **启动命令** | `python -m sglang.launch_server` | `vllm serve` 或 `python -m vllm.entrypoints.openai.api_server` |
| **默认端口** | 30000 | 8000 |
| **多模态支持** | 支持（视觉语言模型） | 支持（视觉语言模型） |
| **量化支持** | FP8, INT4, GPTQ, AWQ 等 | FP8, INT4, GPTQ, AWQ, SqueezeLLM 等 |

## 通俗理解两者的区别

### vLLM 像一个高效的「推理工厂」

vLLM 的核心创新是 **PagedAttention**，类比操作系统的「虚拟内存分页」：
- 传统推理框架把每个请求的 KV Cache 当作连续内存块 → 容易浪费显存
- PagedAttention 把 KV Cache 切成固定大小的「页」，按需分配 → 显存利用率大幅提升
- 就像工厂里的仓库改用标准化货架，空间利用率更高了

### SGLang 像一个「推理工厂 + 智能调度中心」

SGLang 的核心创新是 **RadixAttention** + **前端语言**：
- **RadixAttention（基数树前缀缓存）**：当多个请求共享相同前缀时（如相同的 system prompt），自动复用已计算的 KV Cache → 避免重复计算
  - 类比：如果 100 封信的开头都是「尊敬的客户您好」，只写一次开头，后面各自不同
- **前端语言（SGLang DSL）**：提供编程接口来编排复杂的 LLM 调用链（如 fork/join、选择、约束生成）
  - 类比：vLLM 像一条生产线，SGLang 除了生产线还配了一个调度系统，能协调多条线的工作

### 一句话总结

> **vLLM 专注「推理引擎」做到极致，SGLang 在此基础上多了「前缀缓存」和「编排语言」两张牌。**

## 实际对比：启动服务

下面展示两个框架的启动命令对比，并实际启动 SGLang 进行演示。

In [ ]:
# === 实际对比：启动服务 ===
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求模块

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # 定义使用的模型 ID

# 对比启动命令
print("=" * 60)  # 打印分隔线
print("启动命令对比")  # 打印标题
print("=" * 60)  # 打印分隔线

# SGLang 启动命令
sglang_cmd = f"python -m sglang.launch_server --model-path {MODEL_ID} --host 127.0.0.1 --port 30000"  # SGLang 命令
print(f"\n【SGLang 启动命令】")  # 打印 SGLang 标题
print(f"  {sglang_cmd}")  # 输出 SGLang 命令
print(f"  默认端口: 30000")  # 输出默认端口

# vLLM 启动命令
vllm_cmd = f"vllm serve {MODEL_ID} --host 127.0.0.1 --port 8000"  # vLLM 命令
print(f"\n【vLLM 启动命令】")  # 打印 vLLM 标题
print(f"  {vllm_cmd}")  # 输出 vLLM 命令
print(f"  默认端口: 8000")  # 输出默认端口

# 共同点
print(f"\n【共同点】")  # 打印共同点标题
print("  - 都以 HTTP 服务形式运行")  # 共同点 1
print("  - 都支持 --host 和 --port 参数")  # 共同点 2
print("  - 都兼容 OpenAI API 格式")  # 共同点 3
print("  - 都支持 --model-path 指定模型")  # 共同点 4

# 实际启动 SGLang
print(f"\n{'=' * 60}")  # 打印分隔线
print("正在启动 SGLang 服务进行演示...")  # 打印启动提示

launch_cmd = [  # 构建启动命令列表
    "python", "-m", "sglang.launch_server",  # 使用 sglang 启动服务
    "--model-path", MODEL_ID,  # 指定模型
    "--host", "127.0.0.1",  # 监听地址
    "--port", "30000",  # 监听端口
]

process = subprocess.Popen(  # 启动后台进程
    launch_cmd,  # 传入命令
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.STDOUT,  # 合并标准错误
    text=True  # 文本模式
)

# 等待服务就绪
BASE_URL = "http://127.0.0.1:30000"  # 服务地址
max_wait = 180  # 最大等待秒数
start_time = time.time()  # 记录开始时间

while time.time() - start_time < max_wait:  # 在超时范围内循环
    try:
        resp = requests.get(f"{BASE_URL}/v1/models", timeout=5)  # 尝试请求模型列表
        if resp.status_code == 200:  # 如果成功
            elapsed = time.time() - start_time  # 计算耗时
            print(f"\nSGLang 服务已就绪! (等待了 {elapsed:.1f} 秒)")  # 输出就绪信息
            break  # 跳出循环
    except requests.ConnectionError:  # 连接失败
        pass  # 继续等待
    time.sleep(3)  # 每 3 秒重试
    print(".", end="", flush=True)  # 打印进度点
else:
    print(f"\n警告: 等待 {max_wait} 秒后服务仍未就绪")  # 超时警告

## 实际对比：API 调用

SGLang 和 vLLM 都兼容 OpenAI API，客户端代码几乎完全相同。**唯一的区别就是 `base_url` 中的端口号不同。**

这意味着你可以在两个框架之间**无缝切换**，只需修改一行配置。

In [ ]:
# === 实际对比：API 调用 ===
from openai import OpenAI  # 从 openai 包导入客户端
import time  # 导入时间模块

# 对比：两个框架使用完全相同的客户端代码
print("=" * 60)  # 打印分隔线
print("API 调用对比")  # 打印标题
print("=" * 60)  # 打印分隔线

# SGLang 客户端配置
print("\n【SGLang 客户端配置】")  # 打印 SGLang 配置标题
print('  client = OpenAI(base_url="http://127.0.0.1:30000/v1", api_key="EMPTY")')  # 输出配置代码

# vLLM 客户端配置
print("\n【vLLM 客户端配置】")  # 打印 vLLM 配置标题
print('  client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="EMPTY")')  # 输出配置代码

print("\n→ 唯一区别：端口号 30000 vs 8000")  # 指出唯一差异
print("→ 后续的 API 调用代码完全一致！")  # 强调一致性

# 实际调用 SGLang 进行演示
print(f"\n{'=' * 60}")  # 打印分隔线
print("使用 SGLang 进行实际调用演示")  # 打印演示标题
print("=" * 60)  # 打印分隔线

client = OpenAI(  # 创建 OpenAI 客户端（连接 SGLang）
    base_url="http://127.0.0.1:30000/v1",  # SGLang 服务地址
    api_key="EMPTY"  # 不需要真实 API Key
)

# 发送聊天请求（这段代码对 SGLang/vLLM 通用）
start_time = time.time()  # 记录请求开始时间
try:
    response = client.chat.completions.create(  # 发送聊天补全请求
        model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型
        messages=[  # 设置对话消息列表
            {"role": "system", "content": "你是一个技术专家"},  # 系统角色消息
            {"role": "user", "content": "请用一句话对比 SGLang 和 vLLM"}  # 用户问题
        ],
        max_tokens=150  # 限制最大生成 token 数
    )
    elapsed = time.time() - start_time  # 计算请求耗时
    
    print(f"\n模型回复: {response.choices[0].message.content}")  # 输出模型回复
    print(f"请求耗时: {elapsed:.2f} 秒")  # 输出耗时
    print(f"Token 用量: 输入={response.usage.prompt_tokens}, 输出={response.usage.completion_tokens}")  # 输出 token 统计
except Exception as e:  # 捕获异常
    print(f"请求失败: {e}")  # 输出错误

# 总结 API 对比
print(f"\n{'=' * 60}")  # 打印分隔线
print("API 对比总结")  # 打印总结标题
print("=" * 60)  # 打印分隔线
print("相同点:")  # 打印相同点标题
print("  ✓ 都使用 OpenAI SDK")  # 相同点 1
print("  ✓ 都支持 /v1/chat/completions 接口")  # 相同点 2
print("  ✓ 都支持 /v1/completions 接口")  # 相同点 3
print("  ✓ 都支持流式输出 (stream=True)")  # 相同点 4
print("  ✓ 都支持 /v1/models 列出可用模型")  # 相同点 5
print("\n不同点:")  # 打印不同点标题
print("  △ 默认端口不同（SGLang: 30000, vLLM: 8000）")  # 不同点 1
print("  △ SGLang 额外支持 /generate 原生接口")  # 不同点 2
print("  △ 自定义扩展参数名称可能不同")  # 不同点 3

## 何时选择 SGLang？何时选择 vLLM？

### 选择 SGLang 的场景

1. **多轮对话 / 共享 System Prompt**
   - RadixAttention 自动缓存共同前缀，多用户共享相同 system prompt 时效果显著
   - 例：客服机器人的 system prompt 长达 2000 token，100 个用户共享时只计算一次

2. **结构化输出需求**
   - 原生支持 JSON Schema / 正则表达式约束解码
   - 保证模型输出 100% 符合指定格式

3. **复杂 LLM 编排**
   - SGLang DSL 前端语言支持 fork/join、选择、约束等操作
   - 适合需要多步推理、分支逻辑的应用

4. **前缀密集型工作负载**
   - 如：Few-shot 学习（大量相同示例前缀 + 不同查询后缀）

### 选择 vLLM 的场景

1. **追求稳定生态**
   - vLLM 社区更大、Star 更多、文档更完善
   - 企业级用户更多，踩过的坑和解决方案也更多

2. **广泛的模型支持**
   - vLLM 支持的模型架构列表更长
   - 冷门模型更可能在 vLLM 上找到支持

3. **简单部署场景**
   - 单模型、无特殊前缀缓存需求的标准部署
   - 不需要结构化输出或复杂编排

4. **与现有工具链集成**
   - LangChain、LlamaIndex 等框架对 vLLM 的集成更早、更成熟

## 性能差异概述

性能对比需要注意以下几点：

### 基准测试结果因工作负载而异

- **前缀密集型负载**（大量请求共享前缀）：SGLang 的 RadixAttention 带来显著优势，吞吐量可高出数倍
- **通用推理负载**（每个请求独立）：两者性能接近，vLLM 在某些场景下略优
- **高并发小请求**：两者的连续批处理机制都能很好地处理

### 两者都在快速迭代

- SGLang 和 vLLM 都在积极开发中，每个版本都有显著的性能提升
- 今天的基准测试结果可能在下个月就不适用了
- 建议在 **自己的实际工作负载** 上测试，而非依赖第三方 benchmark

### 关键性能指标对比

| 指标 | SGLang 优势场景 | vLLM 优势场景 |
|------|----------------|---------------|
| 首 token 延迟 | 有前缀缓存命中时极低 | 无前缀场景下两者接近 |
| 吞吐量 | 前缀共享场景大幅领先 | 通用场景下两者接近 |
| 显存效率 | 前缀复用节省显存 | PagedAttention 减少碎片 |
| 长序列推理 | 两者都支持 | 两者都支持 |

In [ ]:
# === 清理资源 ===
import subprocess  # 导入子进程模块

print("正在清理 SGLang 服务进程...")  # 打印清理提示

# 终止本 Notebook 启动的进程
try:
    process.terminate()  # 发送终止信号给服务进程
    process.wait(timeout=10)  # 等待进程结束，超时 10 秒
    print(f"进程 {process.pid} 已终止")  # 输出终止确认
except NameError:  # 如果 process 变量未定义
    print("未找到活跃的服务进程变量")  # 提示未找到进程
except Exception as e:  # 捕获其他异常
    print(f"终止进程时出错: {e}")  # 输出错误信息

# 兜底：杀掉所有 sglang 相关进程
result = subprocess.run(  # 执行 pkill 命令
    ["pkill", "-f", "sglang.launch_server"],  # 按名称匹配并终止 sglang 进程
    capture_output=True, text=True  # 捕获输出
)
print("已尝试清理所有 SGLang 相关进程")  # 输出清理完成提示
print("GPU 显存将在几秒后释放")  # 提示显存释放

## 总结与建议

### 核心结论

1. **两者都是优秀的开源推理框架**，不存在绝对的「谁更好」
2. **SGLang 的独特优势**在于 RadixAttention 前缀缓存和前端编排语言，适合多轮对话、结构化输出等场景
3. **vLLM 的独特优势**在于更成熟的生态和更广泛的模型支持，适合追求稳定性的生产部署
4. **两者 API 兼容**，切换成本极低（只改端口号）

### 实践建议

- **初学者**：两个都试试，先从 API 调用层面体验，感受差异
- **生产部署**：根据实际工作负载做 benchmark，用数据说话
- **长期关注**：两个项目都在快速迭代，持续关注 GitHub 获取最新进展
  - SGLang: https://github.com/sgl-project/sglang
  - vLLM: https://github.com/vllm-project/vllm

## 常见问题排查

### 问题 1：SGLang 和 vLLM 安装到同一环境后依赖冲突

**现象：** 安装其中一个后，另一个报 `ImportError` 或版本不兼容错误。

**可能原因：**
- 两者依赖的 PyTorch、Triton 或其他底层库版本不同
- pip 解析依赖时自动降级/升级了某些包

**解决：** 使用独立的虚拟环境分别安装：
```bash
# 环境 1：SGLang
python -m venv sglang_env
source sglang_env/bin/activate
pip install "sglang[all]" "openai>=1.0.0"

# 环境 2：vLLM
python -m venv vllm_env
source vllm_env/bin/activate
pip install vllm "openai>=1.0.0"
```

### 问题 2：API 响应格式在两个框架间有细微差异

**现象：** 同一段客户端代码在 SGLang 正常运行，在 vLLM 报字段缺失（或反过来）。

**可能原因：**
- 虽然都遵循 OpenAI 格式，但各自的扩展字段可能不同
- 某些可选字段一方返回而另一方不返回

**解决：** 只使用 OpenAI 标准规范中明确定义的字段（如 `choices[0].message.content`、`usage.prompt_tokens` 等），避免依赖框架特有的扩展字段。

```python
# 安全的通用写法
content = response.choices[0].message.content  # 标准字段，两者都支持
tokens = response.usage.prompt_tokens  # 标准字段，两者都支持
```

---

### 结语

SGLang 和 vLLM 代表了开源 LLM 推理领域的两条技术路线。理解它们的核心差异，能帮助你在实际项目中做出更明智的技术选型。无论选择哪个，你都能获得高性能的推理体验——而且切换成本极低。